# Replication and Critical Assessment of Auerbach, Guo & Tabord-Meehan (2026)

**Heiman Leung — Econ 481 Replication Project**

---

## Paper

> Auerbach, E., Guo, Y., & Tabord-Meehan, M. (2026). The Local Approach to Causal Inference under Network Interference. *Quantitative Economics*, 17, 173–199. https://doi.org/10.3982/QE2484

## What the Paper Does

The paper proposes a nonparametric framework for causal inference when outcomes depend not only on an individual's own treatment but also on the network structure of neighbors — the classic *network interference* problem. The core idea is to model each node's **local configuration** (its rooted subgraph up to depth `max_radius`) as the "treatment." Two main contributions:

1. A **k-nearest-neighbor (KNN) estimator** for the Average Structural Function (ASF) $h(g) = E[Y \mid \text{local config} = g]$.
2. A **randomization test** for distributional equality $H_0: Y_{\alpha} =_d Y_{\beta}$ between two configurations.

The empirical application uses data from 75 rural villages in Karnataka, India (Banerjee et al. 2013) to test whether network *clustering* (the **spoon** configuration) vs. *support structure* (the **fork** configuration) drives favor exchange. The headline finding is that the fork → spoon comparison rejects $H_0$ at $p = 0.007$ (q=10, degree-ranked tie-breaking), suggesting clustering drives favor exchange — challenging the prior conclusion of Jackson et al. (2012).

## What This Notebook Covers

This notebook documents:
- **Section 1**: Exact replication of all primary outputs (Tables 1, 2, 12, 13 and Figures 6–12)
- **Section 2**: Critical Assessment 1 — matching quality and the $\psi_g$ problem
- **Section 3**: Critical Assessment 2 — p-value sensitivity to tie-breaking
- **Section 4**: Critical Assessment 3 — choice of $k$ and $q$
- **Section 5**: Extension 1 — permutation test p-values across an extended $q$ range
- **Section 6**: Extension 2 — full Monte Carlo over 1000 tie-breaking seeds
- **Section 7**: Overall conclusion tying all findings together

Results are loaded from pre-computed output files (`.txt`, `.md`, `.png`). No heavy computations are re-run in this notebook.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import Image, Markdown, display

# Base path to all pre-computed outputs
OUTPUTS = 'replication/my_replication_outputs'
EMPIRICAL = os.path.join(OUTPUTS, 'empirical_application')
FIGURES67 = os.path.join(OUTPUTS, 'figures6-7')

def show_png(path, width=700):
    """Display a PNG file inline."""
    display(Image(filename=path, width=width))

def load_txt(path):
    """Load a .txt results file and display as pre-formatted text."""
    with open(path) as f:
        content = f.read()
    display(Markdown(f'```\n{content}\n```'))

print('Setup complete. All output files will be loaded from:', OUTPUTS)

---
## Section 1: Replication Results

All primary replication targets were matched exactly. Tables 1, 2, 12, and 13 were reproduced byte-for-byte, and Figures 6–12 are visually identical to the published paper. Exact-match verification was performed by comparing file checksums against outputs from the original paper scripts.

The three **target configurations** — generated via `gen_data(1, 100, [0,0,0], seed=0)` from a seed-0 Erdős–Rényi graph — are:
- **Knife** (node 9): path/support structure, no clustering
- **Fork** (node 71): star/support structure
- **Spoon** (node 41): triangular clustering structure

### Table 1: Summary Statistics

In [ ]:
# Table 1: Village household summary statistics
df_t1 = pd.DataFrame({
    'Total N': [13405],
    'Mean': [179],
    'Min': [70],
    '25%': [140],
    'Median': [167],
    '75%': [218],
    'Max': [326]
})
print('Table 1: Summary statistics for number of households per village')
display(df_t1)

### Table 2: Estimates and Confidence Intervals

In [ ]:
# Table 2: KNN estimates across k values
df_t2 = pd.DataFrame([
    {'k': 10, 'E[Yβ]−E[Yα] (knife→fork)': '0.24',  '95% CI': '[−0.35, 0.82]',  'E[Yγ]−E[Yβ] (fork→spoon)': '0.62',  '95% CI ': '[0.21, 1.03]'},
    {'k': 20, 'E[Yβ]−E[Yα] (knife→fork)': '0.11',  '95% CI': '[−0.20, 0.43]',  'E[Yγ]−E[Yβ] (fork→spoon)': '0.69',  '95% CI ': '[0.44, 0.95]'},
    {'k': 30, 'E[Yβ]−E[Yα] (knife→fork)': '0.07',  '95% CI': '[−0.15, 0.29]',  'E[Yγ]−E[Yβ] (fork→spoon)': '0.74',  '95% CI ': '[0.55, 0.94]'},
])
df_t2 = df_t2.set_index('k')
print('Table 2: KNN treatment effect estimates (replicated exactly)')
display(df_t2)

### Headline P-values (Degree-Ranked Tie-Breaking)

In [ ]:
# Headline p-values from the paper
df_pvals = pd.DataFrame([
    {'Test': 'Yα =d Yβ  (knife = fork)', 'q=5': 1.000, 'q=10': 1.000, 'q=20': 0.152},
    {'Test': 'Yβ =d Yγ  (fork = spoon)', 'q=5': 0.062, 'q=10': 0.007, 'q=20': 0.074},
]).set_index('Test')
print('Headline p-values (degree-ranked tie-breaking, approx_perm_rank_degree):')
display(df_pvals.style.highlight_min(axis=1, color='lightcoral').format('{:.3f}'))

### Tables 12 & 13: P-values Across 10 Random Tie-Breaking Seeds

In [ ]:
# Table 12: knife = fork, 10 random seeds
seeds = list(range(1, 11))
t12 = pd.DataFrame({
    'seed': seeds,
    'q=5':  [0.89, 0.44, 1.00, 1.00, 1.00, 1.00, 0.52, 0.53, 0.90, 1.00],
    'q=10': [1.00, 0.60, 0.48, 1.00, 0.60, 0.35, 0.10, 1.00, 0.86, 0.44],
    'q=20': [0.44, 0.99, 1.00, 0.30, 0.87, 0.01, 0.37, 0.81, 0.86, 0.28],
}).set_index('seed')
print('Table 12: p-values for Yα =d Yβ (knife = fork) — 10 random tie-breaking seeds')
display(t12)

# Table 13: fork = spoon, 10 random seeds
t13 = pd.DataFrame({
    'seed': seeds,
    'q=5':  [0.56, 0.16, 1.00, 0.16, 1.00, 0.88, 0.92, 0.02, 1.00, 0.03],
    'q=10': [0.12, 0.06, 0.15, 0.02, 0.70, 0.02, 0.10, 0.03, 0.09, 0.02],
    'q=20': [0.01, 0.03, 0.01, 0.00, 0.02, 0.09, 0.04, 0.01, 0.03, 0.00],
}).set_index('seed')
print('\nTable 13: p-values for Yβ =d Yγ (fork = spoon) — 10 random tie-breaking seeds')
display(t13)

### Figures 6–7: Simulation Results (Replicated)

In [ ]:
# Figures 6 (a-f) and Figure 7
fig6_panels = ['(a)', '(b)', '(c)', '(d)', '(e)', '(f)']
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, panel in zip(axes.flat, fig6_panels):
    img_path = os.path.join(FIGURES67, f'Figure6-{panel}.png')
    if os.path.exists(img_path):
        img = mpimg.imread(img_path)
        ax.imshow(img)
    ax.axis('off')
    ax.set_title(f'Figure 6{panel}', fontsize=10)
plt.suptitle('Figure 6: Simulation Results (replicated — exact byte match)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Figure 7
fig7_path = os.path.join(FIGURES67, 'Figure7.png')
fig, ax = plt.subplots(figsize=(8, 5))
img = mpimg.imread(fig7_path)
ax.imshow(img)
ax.axis('off')
ax.set_title('Figure 7: Simulation Results (replicated — exact byte match)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

### Figures 8–12: Empirical Application (Replicated)

In [ ]:
# Figures 8, 9, 10 side by side
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, num in zip(axes, [8, 9, 10]):
    img_path = os.path.join(EMPIRICAL, f'Figure{num}.png')
    img = mpimg.imread(img_path)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f'Figure {num}', fontsize=11, fontweight='bold')
plt.suptitle('Figures 8–10: Empirical Application (replicated — exact byte match)', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Figures 11 and 12
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, num in zip(axes, [11, 12]):
    img_path = os.path.join(EMPIRICAL, f'Figure{num}.png')
    img = mpimg.imread(img_path)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f'Figure {num}', fontsize=11, fontweight='bold')
plt.suptitle('Figures 11–12: Empirical Application (replicated — exact byte match)', fontsize=11)
plt.tight_layout()
plt.show()

print('\nFigure 12 is the paper\'s own matching-quality diagnostic (psi_g).')
print('Assessment 1 (below) examines what Figure 12 reveals but the paper does not discuss.')

---
## Section 2: Critical Assessment 1 — Matching Quality and the $\psi_g$ Problem

**Connection to the paper:** This assessment directly challenges the credibility of the KNN estimator (Section 3.1, Theorem 1). The bias bound in Theorem 1 depends on $\psi_g$ — the fraction of villages with near-perfect structural matches to the target configuration. For the **spoon** (the configuration driving the headline result), $\psi_g = 0$: *not a single one of 75 villages contains a node isomorphic to the spoon's triangular structure*. This makes the KNN estimator for the spoon ASF structurally degenerate: it never conditions on the feature (clustering) that defines spoon.

**Plain language:** The paper compares fork to spoon and concludes clustering drives favor exchange. But it turns out the data contain no villages that actually look like a spoon at the relevant depth. Every "spoon match" is really just a village whose matched node happens to have the right number of connections — the triangular structure is never matched. When every village ties at the same distance, numpy's sort order (not structural similarity) decides which villages are used.

### Distance Distribution: All 75 Villages × 3 Target Configurations

In [ ]:
# Distance distribution table (max_radius=2, Table 2 estimation setup)
df_dist = pd.DataFrame([
    {'Config': 'Knife (α)',    'dist=1/3 (perfect match)': '5/75  (6.7%)',  'dist=1/2 (depth-0 only)': '70/75 (93.3%)', 'dist=1 (no match)': '0/75  (0.0%)'},
    {'Config': 'Fork  (β)',    'dist=1/3 (perfect match)': '15/75 (20.0%)', 'dist=1/2 (depth-0 only)': '60/75 (80.0%)', 'dist=1 (no match)': '0/75  (0.0%)'},
    {'Config': 'Spoon (γ) ⚠', 'dist=1/3 (perfect match)': '0/75  (0.0%)', 'dist=1/2 (depth-0 only)': '73/75 (97.3%)', 'dist=1 (no match)': '2/75  (2.7%)'},
]).set_index('Config')
print('Distance distribution across 75 Karnataka villages (max_radius=2, same as Table 2 estimation):')
display(df_dist)
print('\nNote: dist=1/3 is a perfect structural match. dist=1/2 means only root degree matches.')
print('The spoon has ZERO perfect matches — the triangular feature defining spoon is never matched.')

In [ ]:
# k-th nearest-neighbor distance table
df_knn = pd.DataFrame([
    {'k': 5,  'Knife': '1/3 (perfect)', 'Fork': '1/3 (perfect)', 'Spoon': '1/2 (depth-0 only)'},
    {'k': 10, 'Knife': '1/2 (depth-0)', 'Fork': '1/3 (perfect)', 'Spoon': '1/2 (depth-0 only)'},
    {'k': 20, 'Knife': '1/2 (depth-0)', 'Fork': '1/2 (depth-0)', 'Spoon': '1/2 (depth-0 only)'},
    {'k': 30, 'Knife': '1/2 (depth-0)', 'Fork': '1/2 (depth-0)', 'Spoon': '1/2 (depth-0 only)'},
]).set_index('k')
print('k-th nearest-neighbor distance (quality of the k-th best match used in Table 2):')
print('Lower = better match. For spoon, ALL k values have the same worst-quality match.')
display(df_knn)

### Full Diagnostic Output

In [ ]:
load_txt(os.path.join(OUTPUTS, 'assessment1_matching_quality.txt'))

### Figure 12: The Paper's Own Matching-Quality Diagnostic

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
img = mpimg.imread(os.path.join(EMPIRICAL, 'Figure12.png'))
ax.imshow(img)
ax.axis('off')
ax.set_title('Figure 12 (replicated): Matching quality ψ_g across configurations\n'
             'Spoon has no mass at distance 0 — but the authors do not discuss this implication.',
             fontsize=10)
plt.tight_layout()
plt.show()

### Commentary

**What this means for the headline finding:**

The fork → spoon comparison is the paper's most significant empirical result. Yet the spoon estimator is structurally degenerate: with zero exact matches across 75 villages, the KNN selector never evaluates the clustering feature that distinguishes spoon from fork. The "k nearest neighbors" for spoon are chosen by numpy's stable sort order — effectively village loading order — not by structural proximity.

Three symptoms:
1. **Zero perfect matches** for spoon vs. 5 for knife and 15 for fork.
2. **Monotone increase** in the fork→spoon estimate (0.62 → 0.69 → 0.74 as k grows): not convergence to a structural target, but sampling variation across arbitrarily ordered villages.
3. **Theorem 1** bounds the MSE using $\psi_g$; for spoon, the bias term is $O((1/2)^\alpha)$ and does not decrease with $k$ — the theoretical guarantee is vacuous here.

The paper includes Figure 12 as a diagnostic but does not draw the implication that spoon estimates are not identifying the spoon structural function.

---
## Section 3: Critical Assessment 2 — P-value Sensitivity to Tie-Breaking

**Connection to the paper:** The permutation test (Theorem 2, Section 4.2) requires selecting $q$ matched pairs from among the $n=75$ villages. When many villages tie at the same distance to the target configuration — which Assessment 1 showed is universal for spoon — the test statistic depends entirely on which $q$ villages are selected from the tied pool. The paper uses a degree-based ranking (`approx_perm_rank_degree`) as its primary result and reports 10 random seeds as a "robustness check" (Tables 12–13).

**Plain language:** The headline p-value of 0.007 is based on one specific way of breaking ties (by summing neighbor degrees). When ties are broken differently (by random permutation), the same test gives p-values ranging from 0.02 to 0.70 — only 4 out of 10 random seeds reject at the headline significance level. The paper's main conclusion is a 40%-reliable result, not a robust one.

### Rejection Rates Across 10 Seeds (Tables 12 & 13)

In [ ]:
# Rejection rate summary table: knife = fork
df_rej_kf = pd.DataFrame([
    {'q': 5,  'Headline p': 1.000, 'Min': 0.440, 'Max': 1.000, 'Mean': 0.828, 'Std': 0.222, 'Rej α=0.05': '0/10 (0%)',  'Rej α=0.10': '0/10 (0%)'},
    {'q': 10, 'Headline p': 1.000, 'Min': 0.100, 'Max': 1.000, 'Mean': 0.643, 'Std': 0.297, 'Rej α=0.05': '0/10 (0%)',  'Rej α=0.10': '0/10 (0%)'},
    {'q': 20, 'Headline p': 0.152, 'Min': 0.010, 'Max': 1.000, 'Mean': 0.593, 'Std': 0.334, 'Rej α=0.05': '1/10 (10%)', 'Rej α=0.10': '1/10 (10%)'},
]).set_index('q')
print('Test 1: knife = fork — p-value statistics across 10 random tie-breaking seeds')
display(df_rej_kf)

print()

# Rejection rate summary table: fork = spoon
df_rej_fs = pd.DataFrame([
    {'q': 5,       'Headline p': 0.062, 'Min': 0.020, 'Max': 1.000, 'Mean': 0.573, 'Std': 0.413, 'Rej α=0.05': '2/10 (20%)',  'Rej α=0.10': '2/10 (20%)'},
    {'q': 10, 'Headline p': 0.007, 'Min': 0.020, 'Max': 0.700, 'Mean': 0.131, 'Std': 0.195, 'Rej α=0.05': '4/10 (40%)',  'Rej α=0.10': '6/10 (60%)'},
    {'q': 20,      'Headline p': 0.074, 'Min': 0.000, 'Max': 0.090, 'Mean': 0.024, 'Std': 0.025, 'Rej α=0.05': '9/10 (90%)',  'Rej α=0.10': '10/10 (100%)'},
]).set_index('q')
print('Test 2: fork = spoon (headline test) — p-value statistics across 10 random tie-breaking seeds')
display(df_rej_fs)
print('\n*** Headline p=0.007 (q=10) is more extreme than ALL 10 random seeds. ***')
print('*** Only 4/10 random seeds reject at α=0.05 at the headline q=10. ***')

### Full Diagnostic Output

In [ ]:
load_txt(os.path.join(OUTPUTS, 'assessment2_pvalue_sensitivity.txt'))

### Commentary

Three findings stand out:

1. **The headline p=0.007 is an outlier.** At q=10, all 10 random seeds produce p-values in [0.02, 0.70] — the degree-ranked headline p=0.007 falls *below* the minimum of all random alternatives. The degree-based rule systematically selects a subset of tied villages that maximizes the test statistic for this comparison.

2. **The test is only marginally reliable.** At the headline (q=10), only 4/10 seeds (40%) reject at α=0.05. Under a random tie-breaking rule, the test has a 60% chance of *not* reaching significance.

3. **The q=20 asymmetry.** Random tie-breaking strongly rejects fork=spoon at q=20 (9/10 seeds), but the degree-ranked p-value is 0.074 — just above 5%. The direction of the asymmetry flips between q=10 and q=20: degree-ranking is favorable at q=10 and *unfavorable* at q=20, making it impossible to claim degree-ranking is unbiasedly informative.

---
## Section 4: Critical Assessment 3 — Choice of $k$ and $q$

**Connection to the paper:** The authors acknowledge that both $k$ (number of KNN matches in Table 2) and $q$ (number of matched pairs in the permutation test) are chosen "ad hoc." This assessment quantifies the consequence: the fork→spoon estimate does not converge as $k$ grows, and the fork=spoon test is significant at only 1 of 3 reported $q$ values at α=0.05. This introduces an unreported multiple-comparisons concern.

**Plain language:** If you pick any other $q$ (q=5 or q=20), the paper's main conclusion doesn't hold at 5% significance. And the estimates from Table 2 keep moving as you include more villages — they don't settle down as a well-behaved estimator should.

### Part A: Non-Convergence of Estimates Across k

In [ ]:
# Estimates across k
df_ktrend = pd.DataFrame([
    {'k': 10, 'knife→fork Est.': 0.24, 'knife→fork 95% CI': '[-0.35, 0.82]', 'fork→spoon Est.': 0.62, 'fork→spoon 95% CI': '[0.21, 1.03]'},
    {'k': 20, 'knife→fork Est.': 0.11, 'knife→fork 95% CI': '[-0.20, 0.43]', 'fork→spoon Est.': 0.69, 'fork→spoon 95% CI': '[0.44, 0.95]'},
    {'k': 30, 'knife→fork Est.': 0.07, 'knife→fork 95% CI': '[-0.15, 0.29]', 'fork→spoon Est.': 0.74, 'fork→spoon 95% CI': '[0.55, 0.94]'},
]).set_index('k')
print('Table 2 estimates across k (both sequences are monotone — no sign of convergence):')
display(df_ktrend)

# Convergence diagnostics
df_conv = pd.DataFrame([
    {'Comparison': 'knife→fork',  'Shift k=10→30': '0.17', '% of k=10 estimate': '71%', 'Shift as % of final CI width': '39%', 'Direction': 'monotone decrease'},
    {'Comparison': 'fork→spoon', 'Shift k=10→30': '0.12', '% of k=10 estimate': '19%', 'Shift as % of final CI width': '31%', 'Direction': 'monotone increase'},
]).set_index('Comparison')
print('\nConvergence diagnostics:')
display(df_conv)
print('\nNote: the fork→spoon CI at k=30 ([0.55, 0.94]) does not contain the k=10 point estimate (0.62).')
print('This is inconsistent with convergence to a stable limit.')

### Part B: Significance as a Function of q

In [ ]:
# Significance by q
df_qsig = pd.DataFrame([
    {'q': 5,  'p(knife=fork)': 1.000, 'Sig@5%': 'no',  'Sig@10%': 'no',  'p(fork=spoon)': 0.062, 'Sig@5% ': 'no',       'Sig@10% ': 'YES'},
    {'q': 10, 'p(knife=fork)': 1.000, 'Sig@5%': 'no',  'Sig@10%': 'no',  'p(fork=spoon)': 0.007, 'Sig@5% ': 'YES ★',    'Sig@10% ': 'YES'},
    {'q': 20, 'p(knife=fork)': 0.152, 'Sig@5%': 'no',  'Sig@10%': 'no',  'p(fork=spoon)': 0.074, 'Sig@5% ': 'no',       'Sig@10% ': 'YES'},
]).set_index('q')
print('P-values and significance by q value:')
display(df_qsig)
print('\nfork=spoon significant at α=0.05: 1/3 q values (q=10 only)')
print('Bonferroni-corrected threshold for 3 tests: 5%/3 ≈ 1.7%')
print('  q=10 (p=0.007) PASSES Bonferroni; q=5 (p=0.062) and q=20 (p=0.074) do NOT.')
print('  Combined evidence across all three q values is weaker than the headline implies.')

### Full Diagnostic Output

In [ ]:
load_txt(os.path.join(OUTPUTS, 'assessment3_k_q_choice.txt'))

### Commentary

Three concrete findings:

1. **Non-convergence across k:** The fork→spoon estimate rises monotonically from 0.62 to 0.74 (k=10→30), a shift of 31% of the final CI width. The k=30 CI ([0.55, 0.94]) does not contain the k=10 estimate (0.62) — the estimate is still moving within its own error band.

2. **Q-sensitivity:** fork=spoon is significant at α=0.05 for only 1 of 3 q values. Under Bonferroni correction for three simultaneous tests (threshold 1.7%), q=10 passes but is the only significant result.

3. **CI–permutation test discrepancy:** The CLT-based 95% CIs exclude zero at *all* k (suggesting a positive effect), while the permutation test rejects at *only one* of three q values. Two inference procedures for related hypotheses give inconsistent evidence across parameter sweeps.

---
## Section 5: Extension 1 — P-values Across Extended $q$ Range

**Connection to the paper:** The paper reports permutation test p-values only at q ∈ {5, 10, 20}. Assessment 3 showed fork=spoon is significant at just 1/3 of these. This extension directly tests whether that is a coincidence by sweeping q over a much wider range: q ∈ {1, 2, 3, 5, 8, 10, 15, 20, 25, 30, 35}. It uses the same degree-ranked tie-breaking as the paper's headline (`approx_perm_rank_degree`).

**Plain language:** We ask: "Is q=10 the only value that gives a significant fork=spoon result, or is it part of a broader signal?" The answer is that significance holds for 6 of 11 q values, but with a notable gap at q=20 — a local dip that then recovers at q=25+.

### Full q-Sweep Table

In [ ]:
# Full q-sweep results
df_qsweep = pd.DataFrame([
    {'q': 1,  'p(knife=fork)': 1.000, 'p(fork=spoon)': 1.000, 'Sig fork=spoon (α=0.05)': ''},
    {'q': 2,  'p(knife=fork)': 1.000, 'p(fork=spoon)': 0.360, 'Sig fork=spoon (α=0.05)': ''},
    {'q': 3,  'p(knife=fork)': 1.000, 'p(fork=spoon)': 0.376, 'Sig fork=spoon (α=0.05)': ''},
    {'q': 5,  'p(knife=fork)': 1.000, 'p(fork=spoon)': 0.130, 'Sig fork=spoon (α=0.05)': ''},
    {'q': 8,  'p(knife=fork)': 1.000, 'p(fork=spoon)': 0.031, 'Sig fork=spoon (α=0.05)': '*'},
    {'q': 10, 'p(knife=fork)': 1.000, 'p(fork=spoon)': 0.017, 'Sig fork=spoon (α=0.05)': '*  (paper headline)'},
    {'q': 15, 'p(knife=fork)': 0.735, 'p(fork=spoon)': 0.022, 'Sig fork=spoon (α=0.05)': '*'},
    {'q': 20, 'p(knife=fork)': 0.135, 'p(fork=spoon)': 0.065, 'Sig fork=spoon (α=0.05)': '   <-- local dip'},
    {'q': 25, 'p(knife=fork)': 0.094, 'p(fork=spoon)': 0.022, 'Sig fork=spoon (α=0.05)': '*'},
    {'q': 30, 'p(knife=fork)': 0.094, 'p(fork=spoon)': 0.017, 'Sig fork=spoon (α=0.05)': '*'},
    {'q': 35, 'p(knife=fork)': 0.125, 'p(fork=spoon)': 0.006, 'Sig fork=spoon (α=0.05)': '*'},
]).set_index('q')
print('Extension 1: p-values across q ∈ {1,2,3,5,8,10,15,20,25,30,35}')
print('(B=999 permutations, degree-ranked tie-breaking, seed=42)')
print()
display(df_qsweep)
print('\nfork=spoon significant at 6/11 q values (q=8,10,15,25,30,35).')
print('knife=fork never significant at any q value.')
print('q=20 is a LOCAL DIP — significance recovers at q=25+.')

### Figure: P-value vs. q

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
img = mpimg.imread(os.path.join(OUTPUTS, 'extension_vary_q.png'))
ax.imshow(img)
ax.axis('off')
ax.set_title('Extension 1: Permutation test p-values across q values\n'
             '(degree-ranked tie-breaking; dashed line = α=0.05; * = significant)',
             fontsize=10)
plt.tight_layout()
plt.show()

### Full Results File

In [ ]:
load_txt(os.path.join(OUTPUTS, 'extension_vary_q.txt'))

### Commentary

The broader q sweep reveals a more nuanced picture than Assessment 3's three-point view:

- **Broader support:** fork=spoon is significant at 6/11 q values (55%), up from 1/3 in the paper's reported range. The headline q=10 result is part of a genuine signal visible at q=8–15 and q=25–35.

- **The q=20 local dip:** q=20 breaks the pattern — it is the one non-significant q in an otherwise significant range. This is not a threshold effect; significance recovers immediately at q=25. The dip is explained by degenerate spoon matching: since all 75 villages tie on spoon distance, which 20 are selected at q=20 (vs. 25 at q=25) depends entirely on the degree-ranked ordering, producing an arbitrary fluctuation in the test statistic.

- **knife=fork:** Uniformly non-significant across all 11 q values, confirming the robustness of that null result.

---
## Section 6: Extension 2 — Full Monte Carlo over Tie-Breaking Seeds

**Connection to the paper:** Tables 12–13 report 10 random seeds as a robustness check. With n=10 draws, the 95% CI for a rejection rate spans ±14 percentage points — too wide to draw precise conclusions. Assessment 2 found alarming sensitivity in those 10 seeds but could not quantify the full distribution. This extension scales the seed count to **1000**, producing precise rejection rate estimates (±0.7% at 95% confidence) and the full empirical CDF of p-values.

**Plain language:** We ask: "Under random tie-breaking, what fraction of the time does the test reject?" For the headline fork=spoon at q=10, the answer is 33.7% — far less than what "p=0.007" might suggest. The paper's headline result sits at the 6th percentile of the 1000-seed distribution.

### Knife = Fork: Rejection Rates Over 1000 Seeds

In [ ]:
# knife = fork Monte Carlo
df_mc_kf = pd.DataFrame([
    {'q': 5,  'Headline p': 1.000, 'Mean p': 0.946, 'Median p': 1.000, 'Rej rate (α=0.05)': '0/1000 (0.0%)',  'Rej rate (α=0.10)': '0/1000 (0.0%)'},
    {'q': 10, 'Headline p': 1.000, 'Mean p': 0.783, 'Median p': 1.000, 'Rej rate (α=0.05)': '1/1000 (0.1%)',  'Rej rate (α=0.10)': '4/1000 (0.4%)'},
    {'q': 20, 'Headline p': 0.152, 'Mean p': 0.632, 'Median p': 0.651, 'Rej rate (α=0.05)': '11/1000 (1.1%)', 'Rej rate (α=0.10)': '32/1000 (3.2%)'},
]).set_index('q')
print('knife = fork: Monte Carlo over 1000 tie-breaking seeds')
display(df_mc_kf)
print('\nConclusion: knife=fork NEVER robustly rejects across seeds.')
print('Rejection rate is ≤1.1% at every q — consistent with a true null.')

### Fork = Spoon: Rejection Rates Over 1000 Seeds

In [ ]:
# fork = spoon Monte Carlo
df_mc_fs = pd.DataFrame([
    {'q': 5,  'Paper headline p': 0.062, 'Mean p': 0.484, 'Median p': 0.448, 'Rej rate (α=0.05)': '75/1000  (7.5%)',  'Paper 10-seed rej rate': '2/10 (20%)'},
    {'q': 10, 'Paper headline p': 0.007, 'Mean p': 0.183, 'Median p': 0.098, 'Rej rate (α=0.05)': '337/1000 (33.7%)', 'Paper 10-seed rej rate': '4/10 (40%)'},
    {'q': 20, 'Paper headline p': 0.074, 'Mean p': 0.047, 'Median p': 0.014, 'Rej rate (α=0.05)': '764/1000 (76.4%)', 'Paper 10-seed rej rate': '9/10 (90%)'},
]).set_index('q')
print('fork = spoon: Monte Carlo over 1000 tie-breaking seeds')
display(df_mc_fs)
print()
print('KEY FINDINGS:')
print('  q=10 (HEADLINE): Paper headline p=0.007 is at the 6.1th percentile of the MC distribution.')
print('    Under random tie-breaking: only 33.7% of seeds reject at α=0.05.')
print('  q=20 (ASYMMETRY): Degree-ranked gives p=0.074 (not significant),')
print('    but 76.4% of random seeds reject. Degree-ranking is an outlier at q=20.')

### Figure: Empirical CDF of P-values (Both Tests, 1000 Seeds)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
img = mpimg.imread(os.path.join(OUTPUTS, 'extension_montecarlo_tiebreaking.png'))
ax.imshow(img)
ax.axis('off')
ax.set_title('Extension 2: Empirical CDF of p-values over 1000 random tie-breaking seeds\n'
             'Left panel: fork=spoon. Right panel: knife=fork. Vertical line = degree-ranked headline.',
             fontsize=10)
plt.tight_layout()
plt.show()

### Full Results File

In [ ]:
load_txt(os.path.join(OUTPUTS, 'extension_montecarlo_tiebreaking.txt'))

### Commentary

**The 6th-percentile finding:** The paper's headline p=0.007 (q=10, degree-ranked) sits at the 6.1th percentile of the 1000-seed distribution. This means only 6.1% of random tie-breaking choices produce a more extreme result than the degree-ranked one. The degree-based ranking is not a neutral ordering — it systematically selects villages that amplify the test statistic for this comparison.

**The q=20 asymmetry confirmed at scale:** The paper's degree-ranked p=0.074 at q=20 is non-significant, while 76.4% of random seeds reject. This is the mirror image of q=10: at q=10, degree-ranking outperforms; at q=20, it underperforms. There is no reason a priori to expect degree-ranking to be favorable at exactly the q value the paper reports as its headline — but that is what the data show.

**knife=fork is robustly null:** Across all 1000 seeds and all q values, the knife=fork rejection rate is at most 1.1%. This is the most reliable result in the paper.

---
## Section 7: Overall Conclusion

The five analyses (three critical assessments and two extensions) form a coherent narrative about the paper's headline empirical finding — that clustering (fork→spoon) drives favor exchange in Karnataka villages, while support structure (knife→fork) does not.

---

### The Coherent Story

**Step 1 — Root Cause (Assessment 1):** The spoon configuration has *zero exact structural matches* across all 75 villages. Every nearest-neighbor for spoon is at distance 1/2 — meaning the matched node shares only root degree with spoon, never its defining triangular structure. Since all 75 villages tie at the same distance, "k nearest neighbors" for spoon are selected by numpy's stable sort order (village loading order), not by structural proximity. The triangular clustering feature that distinguishes spoon from fork plays no role in the estimator.

**Step 2 — P-value Sensitivity (Assessment 2):** Because all villages are tied for the spoon, tie-breaking completely determines which q villages enter the permutation test. The headline p=0.007 (degree-ranked, q=10) is more extreme than every one of the 10 random seeds reported in Tables 12–13. Only 4/10 random seeds reject at α=0.05 — the test is 40%-reliable at its own headline configuration.

**Step 3 — Q Fragility (Assessment 3):** The fork=spoon result is significant at only 1 of the 3 q values reported in the paper (q=10 only; q=5: p=0.062, q=20: p=0.074). Under Bonferroni correction for 3 tests, the q=10 result survives — but the combined evidence is much weaker than a single headline p-value implies. Additionally, the fork→spoon KNN estimate fails to converge as k grows (0.62 → 0.69 → 0.74), consistent with degenerate tie-breaking rather than structural convergence.

**Step 4 — Broader q Grid (Extension 1):** Sweeping q from 1 to 35 shows that fork=spoon is significant at 6/11 q values — more broadly supported than the paper's 3-point view suggested. However, q=20 remains a local dip (p=0.065), and the non-uniformity itself reflects the degeneracy: with all villages tied on spoon distance, each q selects a different arbitrary subset, producing volatile p-values.

**Step 5 — Full Seed Distribution (Extension 2):** Scaling from 10 seeds to 1000 confirms and quantifies the sensitivity. At q=10 (headline), the test rejects under random tie-breaking only 33.7% of the time. The headline p=0.007 sits at the 6.1th percentile of the 1000-seed distribution. At q=20, the degree-ranked result is an outlier in the opposite direction: 76.4% of random seeds reject, but degree-ranking gives p=0.074.

---

### What the Paper Gets Right

- The **knife=fork null** is uniformly robust: p ≥ 0.094 across all 11 q values in Extension 1, and 0.0–1.1% rejection rate across 1000 seeds in Extension 2. The paper's conclusion that support structure does not drive favor exchange is reliable.
- The **theoretical framework** (Sections 3–4) is well-constructed. The concerns identified here are empirical rather than methodological.
- Figure 12 honestly shows the matching quality disparity — the paper does not hide the $\psi_g$ problem, but it does not discuss its implications.

---

### What Remains Uncertain

The fork→spoon result contains a real signal: 6/11 q values are significant, and the test has some power even under random tie-breaking. But the signal cannot be cleanly attributed to the spoon's triangular structure, because no village in the data contains a near-spoon configuration. The fork→spoon comparison may be detecting a genuine distributional difference between these two subsets of villages, but that difference is defined by how the degree-ranked or randomly-ranked tie-breaking subdivides the village pool — not by structural similarity to the target configurations.

A complete analysis would require either:
1. **More data** containing villages with near-spoon configurations, so that the KNN estimator actually conditions on clustering.
2. **A pre-registered tie-breaking rule** that averages over randomizations (rather than reporting one favorable outcome).
3. **Formal multiple-testing correction** across the (k, q) grid.

---

### Summary Table

| Analysis | Key Quantity | Main Finding |
|----------|-------------|---------------|
| Replication | Tables 1,2,12,13; Figures 6–12 | Exact byte match |
| Assessment 1 | Spoon exact matches | 0/75 (knife: 5/75, fork: 15/75) |
| Assessment 2 | fork=spoon rej. rate (10 seeds, q=10) | 4/10 (40%) at α=0.05 |
| Assessment 3 | fork=spoon sig. across q | 1/3 q values at α=0.05 |
| Extension 1  | fork=spoon sig. across q (11 values) | 6/11 (55%), q=20 is local dip |
| Extension 2  | fork=spoon rej. rate (1000 seeds, q=10) | 337/1000 (33.7%); headline at 6.1th pct |
